# MLP Hyperparameter Tuning for Customer Churn

This notebook tunes a Keras MLP classifier with Optuna, mirroring the same TPE-based
search strategy used in `Tabnet_tuning.ipynb`: an `objective()` function trains a model
with early stopping and returns validation AUC, `optuna.create_study(direction="maximize")`
runs the search, and the winning parameters are refit into a final model that is evaluated
on the held-out test set.

The main differences from the TabNet notebook come from the model itself:
- TabNet handles raw/categorical features internally; a plain MLP needs one-hot encoding
  for any categorical columns and feature scaling (`StandardScaler`, fit on train only).
- TabNet exposes `feature_importances_` natively; the MLP doesn't, so importance here is
  computed via permutation importance (AUC drop when a column is shuffled).
- Early stopping during tuning still watches `val_loss` (matching TabNet's
  `eval_metric=["logloss"]`), while the Optuna objective itself returns validation AUC,
  and `direction="maximize"` matches that returned value — not the loss used internally.


In [ ]:
# Optional, run once if the environment does not already contain these packages.
# %pip install optuna tensorflow scikit-learn seaborn matplotlib

from pathlib import Path
import json
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
import seaborn as sns
import tensorflow as tf

from optuna.samplers import TPESampler
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
N_TRIALS = 10
MAX_EPOCHS = 20
PATIENCE = 5
BATCH_SIZE = 2048
ARTIFACT_DIR = Path("mlp_optuna_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"Optuna version: {optuna.__version__}")
print(f"TensorFlow version: {tf.__version__}")
gpus = tf.config.list_physical_devices("GPU")
print(f"TensorFlow device: {'GPU' if gpus else 'CPU'}")

In [ ]:
# Load the supplied split. The fallback makes the notebook portable when opened from the workspace root.
DATA_DIR = Path(r"C:\Users\User\ODL_Labs\Assignment\data")
if not DATA_DIR.exists():
    DATA_DIR = Path("Assignment/data")

train_df = pd.read_csv(DATA_DIR / "train_churn_dataset.csv")
val_df = pd.read_csv(DATA_DIR / "val_churn_dataset.csv")
test_df = pd.read_csv(DATA_DIR / "test_churn_dataset.csv")

target_col = "Churn"

feature_columns = [column for column in train_df.columns if column != target_col]

print(f"Target column: {target_col}")
print(f"Train/validation/test shapes: {train_df.shape}, {val_df.shape}, {test_df.shape}")
print("Target distribution (train):")
print(train_df[target_col].value_counts(normalize=True).sort_index())

In [ ]:
def encode_target(train_frame, val_frame, test_frame, target_column):
    """Encode target labels from the training split and reject unseen labels."""
    classes, train_codes = np.unique(train_frame[target_column], return_inverse=True)
    class_to_code = {value: index for index, value in enumerate(classes)}

    def encode(frame):
        unseen = sorted(set(frame[target_column].dropna()) - set(class_to_code))
        if unseen:
            raise ValueError(f"Unseen target labels in {target_column}: {unseen}")
        if frame[target_column].isna().any():
            raise ValueError(f"Missing target labels found in {target_column}.")
        return frame[target_column].map(class_to_code).to_numpy(dtype=np.int64)

    return encode(train_frame), encode(val_frame), encode(test_frame), classes


def prepare_mlp_features(train_frame, val_frame, test_frame, columns):
    """Fit train-only imputation, one-hot encoding, and scaling for a plain MLP.

    Unlike TabNet, a dense network has no native handling for categorical indices,
    so any categorical columns are one-hot encoded here (fit on the union of splits
    to avoid column-mismatch, but medians/scaler are still fit on train only).
    Every feature is standardized with a StandardScaler fit on the training split.
    """
    train_features = train_frame[columns].copy()
    val_features = val_frame[columns].copy()
    test_features = test_frame[columns].copy()

    categorical_columns = [
        column
        for column in columns
        if (
            pd.api.types.is_object_dtype(train_features[column])
            or isinstance(train_features[column].dtype, pd.CategoricalDtype)
            or pd.api.types.is_bool_dtype(train_features[column])
        )
    ]
    numeric_columns = [column for column in columns if column not in categorical_columns]

    for column in numeric_columns:
        numeric_train = pd.to_numeric(train_features[column], errors="coerce")
        median = numeric_train.median()
        if pd.isna(median):
            raise ValueError(f"Numeric feature '{column}' has no usable training values.")
        for frame in (train_features, val_features, test_features):
            frame[column] = pd.to_numeric(frame[column], errors="coerce").fillna(median)

    if categorical_columns:
        combined = pd.concat(
            [train_features, val_features, test_features],
            keys=["train", "val", "test"],
        )
        combined = pd.get_dummies(combined, columns=categorical_columns, dummy_na=False)
        train_features = combined.loc["train"]
        val_features = combined.loc["val"].reindex(columns=train_features.columns, fill_value=0)
        test_features = combined.loc["test"].reindex(columns=train_features.columns, fill_value=0)

    feature_names = list(train_features.columns)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_features.to_numpy(dtype=np.float64)).astype(np.float32)
    X_val = scaler.transform(val_features.to_numpy(dtype=np.float64)).astype(np.float32)
    X_test = scaler.transform(test_features.to_numpy(dtype=np.float64)).astype(np.float32)

    if not all(np.isfinite(array).all() for array in (X_train, X_val, X_test)):
        raise ValueError("Feature arrays contain non-finite values after preprocessing.")

    return X_train, X_val, X_test, feature_names, scaler, categorical_columns


y_train, y_val, y_test, class_labels = encode_target(
    train_df, val_df, test_df, target_col
)
X_train, X_val, X_test, feature_names, scaler, categorical_columns = prepare_mlp_features(
    train_df, val_df, test_df, feature_columns
)

if len(class_labels) != 2:
    raise ValueError(f"This notebook expects binary churn labels, found {len(class_labels)} classes.")

print(f"Feature matrix shapes: {X_train.shape}, {X_val.shape}, {X_test.shape}")
print(f"Categorical columns one-hot encoded: {categorical_columns}")
print(f"Encoded target classes: {list(class_labels)}")

In [ ]:
def build_mlp(params, input_dim):
    """Create a compiled Keras MLP classifier from an Optuna parameter dictionary."""
    inputs = layers.Input(shape=(input_dim,))
    x = inputs
    for i in range(params["n_layers"]):
        x = layers.Dense(
            params[f"units_l{i}"],
            activation=params["activation"],
            kernel_regularizer=tf.keras.regularizers.l2(params["l2"]),
        )(x)
        if params["batch_norm"]:
            x = layers.BatchNormalization()(x)
        x = layers.Dropout(params["dropout"])(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = models.Model(inputs, outputs)
    optimizer = tf.keras.optimizers.Adam(
        learning_rate=params["learning_rate"],
        weight_decay=params["weight_decay"],
    )
    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(name="auc")],
    )
    return model

In [ ]:
def objective(trial):
    n_layers = trial.suggest_int("n_layers", 1, 4)
    params = {
        "n_layers": n_layers,
        "activation": trial.suggest_categorical("activation", ["relu", "elu"]),
        "dropout": trial.suggest_float("dropout", 0.0, 0.5),
        "l2": trial.suggest_float("l2", 1e-8, 1e-3, log=True),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 5e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-8, 1e-3, log=True),
        "batch_norm": trial.suggest_categorical("batch_norm", [True, False]),
    }
    for i in range(n_layers):
        params[f"units_l{i}"] = trial.suggest_int(f"units_l{i}", 16, 256, step=16)

    tf.keras.backend.clear_session()
    model = build_mlp(params, input_dim=X_train.shape[1])

    # Early stopping watches validation loss, mirroring TabNet's eval_metric=["logloss"].
    # Optuna's objective still returns AUC below -- these are two independent choices.
    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True,
    )
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stop],
        verbose=0,
    )

    validation_probability = model.predict(X_val, verbose=0).ravel()
    validation_auc = roc_auc_score(y_val, validation_probability)
    trial.set_user_attr("validation_auc", float(validation_auc))
    trial.set_user_attr("best_epoch", int(np.argmin(history.history["val_loss"])))
    return validation_auc

print("Objective function is ready.")

### Run the Optuna study

This runs `N_TRIALS` trials of TPE-based search, optimizing validation AUC -- the
exact same strategy as `Tabnet_tuning.ipynb` (same sampler, same `direction="maximize"`,
same pattern of returning AUC from the objective while early stopping watches loss).
Each trial trains an MLP with early stopping on the validation set, so this cell can
take a while depending on `N_TRIALS`, `MAX_EPOCHS`, and hardware.

In [ ]:
sampler = TPESampler(seed=SEED)
study = optuna.create_study(
    direction="maximize",
    sampler=sampler,
    study_name="mlp_churn",
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best_trial = study.best_trial
print(f"Number of finished trials: {len(study.trials)}")
print("Best trial:")
print(f"  Validation AUC: {best_trial.value:.4f}")
print(f"  Best epoch:     {best_trial.user_attrs.get('best_epoch')}")
print("  Params:")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

# Persist study results so the search doesn't need to be repeated.
study_df = study.trials_dataframe()
study_df.to_csv(ARTIFACT_DIR / "optuna_trials.csv", index=False)
with open(ARTIFACT_DIR / "best_params.json", "w") as f:
    json.dump(best_trial.params, f, indent=2)

### Optimization history & parameter importance

Quick diagnostics on the search itself: how the objective improved across trials, and
which hyperparameters mattered most.

In [ ]:
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances

ax = plot_optimization_history(study)
ax.figure.tight_layout()
ax.figure.savefig(ARTIFACT_DIR / "optuna_optimization_history.png", dpi=100, bbox_inches="tight")
plt.show()

ax = plot_param_importances(study)
ax.figure.tight_layout()
ax.figure.savefig(ARTIFACT_DIR / "optuna_param_importances.png", dpi=100, bbox_inches="tight")
plt.show()

### Train the final model with the best hyperparameters

Refit an MLP using the winning trial's parameters. Here early stopping watches
validation AUC directly (mirroring the TabNet notebook's final refit, which switched
`eval_metric` to `["auc"]`), then we evaluate on the untouched test set below.

In [ ]:
best_params = dict(study.best_trial.params)

tf.keras.backend.clear_session()
final_model = build_mlp(best_params, input_dim=X_train.shape[1])

early_stop = EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=PATIENCE,
    restore_best_weights=True,
)
history = final_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1,
)

best_epoch = int(np.argmax(history.history["val_auc"]))
saved_path = ARTIFACT_DIR / "mlp_best_model.keras"
final_model.save(saved_path)
print(f"Best epoch: {best_epoch}")
print(f"Model saved to: {saved_path}")

### Training curves

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(history.history["loss"], label="Training Loss", color="blue")
plt.plot(history.history["val_loss"], label="Validation Loss", color="orange")
plt.title("Training and Validation Loss", fontsize=14)
plt.xlabel("Epochs", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.legend(loc="upper right", fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "training_loss.png", dpi=100, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 6))
plt.plot(history.history["auc"], label="Training AUC", color="blue")
plt.plot(history.history["val_auc"], label="Validation AUC", color="orange")
plt.title("Training and Validation AUC", fontsize=14)
plt.xlabel("Epochs", fontsize=12)
plt.ylabel("AUC", fontsize=12)
plt.legend(loc="lower right", fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "training_auc.png", dpi=100, bbox_inches="tight")
plt.show()

### Evaluate on the held-out test set

In [ ]:
test_probability = final_model.predict(X_test, verbose=0).ravel()
test_prediction = (test_probability > 0.5).astype(int)

report = classification_report(y_test, test_prediction, output_dict=True)
accuracy = report["accuracy"]
precision = report["1"]["precision"]
recall = report["1"]["recall"]
f1 = report["1"]["f1-score"]
test_auc = roc_auc_score(y_test, test_probability)

print("CLASSIFICATION REPORT:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"AUC:       {test_auc:.4f}")

with open(ARTIFACT_DIR / "test_metrics.json", "w") as f:
    json.dump(
        {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "auc": test_auc,
        },
        f,
        indent=2,
    )

### ROC curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, test_probability)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC Curve (AUC = {test_auc:.4f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--", label="Random Classifier")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate", fontsize=12)
plt.ylabel("True Positive Rate", fontsize=12)
plt.title("ROC Curve - MLP (Test Set)", fontsize=14)
plt.legend(loc="lower right", fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "roc_auc_curve.png", dpi=100, bbox_inches="tight")
plt.show()

print(f"AUC Score: {test_auc:.4f}")

### Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, test_prediction)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=True,
    xticklabels=["No Churn", "Churn"],
    yticklabels=["No Churn", "Churn"],
)
plt.title("Confusion Matrix - MLP (Test Set)", fontsize=14, fontweight="bold")
plt.ylabel("True Label", fontsize=12)
plt.xlabel("Predicted Label", fontsize=12)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "confusion_matrix.png", dpi=100, bbox_inches="tight")
plt.show()

### Feature importance

Keras MLPs have no built-in `feature_importances_` the way TabNet or tree-based models
do, so importance is computed here via permutation importance: shuffle one feature
column at a time on the validation set and measure how much AUC drops. A larger drop
means the model relied on that feature more heavily.

In [ ]:
def permutation_importance_auc(model, X, y, names, n_repeats=5, seed=SEED):
    """Permutation importance using AUC drop as the scoring metric."""
    rng = np.random.default_rng(seed)
    baseline_pred = model.predict(X, verbose=0).ravel()
    baseline_auc = roc_auc_score(y, baseline_pred)

    importances = np.zeros(X.shape[1])
    for col in range(X.shape[1]):
        drops = np.empty(n_repeats)
        for r in range(n_repeats):
            X_permuted = X.copy()
            rng.shuffle(X_permuted[:, col])
            permuted_pred = model.predict(X_permuted, verbose=0).ravel()
            drops[r] = baseline_auc - roc_auc_score(y, permuted_pred)
        importances[col] = drops.mean()

    return pd.Series(importances, index=names).sort_values(ascending=False)


importances = permutation_importance_auc(final_model, X_val, y_val, feature_names)

plt.figure(figsize=(8, max(4, 0.35 * len(importances))))
sns.barplot(x=importances.values, y=importances.index, color="steelblue")
plt.title("MLP Permutation Feature Importances (AUC drop)", fontsize=14)
plt.xlabel("Mean AUC decrease when shuffled", fontsize=12)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "feature_importances.png", dpi=100, bbox_inches="tight")
plt.show()

importances.to_frame("importance").to_csv(ARTIFACT_DIR / "feature_importances.csv")
importances

### Summary

All tuning artifacts (Optuna trial log, best hyperparameters, trained model, metrics,
and plots) are written to the `mlp_optuna_artifacts/` directory for later reference.

In [ ]:
print("Artifacts written to:", ARTIFACT_DIR.resolve())
for path in sorted(ARTIFACT_DIR.glob("*")):
    print(" -", path.name)